<!-- NOTEBOOK_METADATA source: "⚠️ Jupyter Notebook" title: "Multimodal Google Gemini Tracing in JS/TS with Langfuse" sidebarTitle: "Google Gemini Multimodal (JS/TS)" logo: "/images/integrations/google_gemini_icon.svg" description: "Learn how to trace multimodal Google Gemini calls (image + text) in JavaScript/TypeScript with Langfuse, including image token cost capture and image rendering in the Langfuse UI." category: "Integrations" -->

# Trace Multimodal Google Gemini Calls with Langfuse (JS/TS)

<a href="https://langfuse.com/integrations/model-providers/google-gemini"><img className="inline" alt="Python" src="https://img.shields.io/badge/Python-3776AB?style=flat&logo=python&logoColor=white" /></a> <a href="https://langfuse.com/integrations/model-providers/google-gemini-js"><img className="inline" alt="JS/TS" src="https://img.shields.io/badge/JS/TS-d4d4d8?style=flat&logo=javascript&logoColor=white" /></a>

This cookbook shows how to trace image-based (multimodal) [Google Gemini](https://ai.google.dev/gemini-api/docs) calls in JavaScript/TypeScript using the [Langfuse JS/TS SDK](https://langfuse.com/docs/observability/sdk/typescript/overview). You will learn how to:

- Capture image token counts from `usageMetadata` for accurate cost tracking
- Include inline images in the Langfuse generation input so they render in the Langfuse UI
- Apply the `flushAsync()` pattern for serverless and Next.js environments

> **What is Google Gemini?** [Google Gemini](https://ai.google.dev/gemini-api/docs/libraries) is Google's family of multimodal generative models. Models like `gemini-2.0-flash` accept text, images, audio, and video as input.

> **What is Langfuse?** [Langfuse](https://langfuse.com) is an open source LLM observability platform that captures inputs (including images), outputs, token usage, latency, and cost for every LLM call.

<!-- STEPS_START -->
## Step 1: Install Dependencies

```bash
npm install @google/generative-ai langfuse
```

> **Note**: This cookbook uses **Deno.js** for execution. For Node.js, the same imports work with standard `npm` packages and `process.env`.

## Step 2: Configure Environment

Set your Langfuse API keys and Google Gemini API key.

In [ ]:
// Langfuse authentication keys
Deno.env.set("LANGFUSE_PUBLIC_KEY", "pk-lf-...");
Deno.env.set("LANGFUSE_SECRET_KEY", "sk-lf-...");

// Langfuse host — 🇪🇺 EU region
// Other regions: 🇺🇸 US: https://us.cloud.langfuse.com, 🇯🇵 Japan: https://jp.cloud.langfuse.com, ⚕️ HIPAA: https://hipaa.cloud.langfuse.com
Deno.env.set("LANGFUSE_BASE_URL", "https://cloud.langfuse.com");

// Google Gemini API key
Deno.env.set("GEMINI_API_KEY", "...");

## Step 3: Initialize Langfuse

In [ ]:
import { Langfuse } from "npm:langfuse";

const langfuse = new Langfuse({
  publicKey: Deno.env.get("LANGFUSE_PUBLIC_KEY"),
  secretKey: Deno.env.get("LANGFUSE_SECRET_KEY"),
  baseUrl: Deno.env.get("LANGFUSE_BASE_URL"),
});

## Step 4: Trace a Multimodal (Image + Text) Call

This example fetches a public image, sends it to Gemini alongside a text prompt, and records the full call in Langfuse — including the image in the generation input so it renders in the Langfuse UI.

> **Image token costs**: Gemini reports image token usage inside `usageMetadata.promptTokenCount`. This count includes both the text and image tokens combined. Pass these values to `generation.end()` so Langfuse can calculate the total cost of multimodal calls.

In [ ]:
import { GoogleGenerativeAI } from "npm:@google/generative-ai";
import { Langfuse } from "npm:langfuse";

const genAI = new GoogleGenerativeAI(Deno.env.get("GEMINI_API_KEY") ?? "");

const MODEL = "gemini-2.0-flash";

// Fetch a sample image and convert it to base64
const imageUrl = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png";
const imageResponse = await fetch(imageUrl);
const imageBuffer = await imageResponse.arrayBuffer();
const imageBase64 = btoa(String.fromCharCode(...new Uint8Array(imageBuffer)));
const mimeType = "image/png";

const textPrompt = "Describe what you see in this image.";

// Create a trace for this multimodal request
const trace = langfuse.trace({
  name: "gemini-multimodal",
  input: { prompt: textPrompt, imageSource: imageUrl },
});

// Include the image in the generation input — Langfuse renders it in the UI
const generation = trace.generation({
  name: "gemini-vision",
  model: MODEL,
  input: [
    {
      role: "user",
      content: [
        {
          type: "image_url",
          image_url: { url: `data:${mimeType};base64,${imageBase64}` },
        },
        { type: "text", text: textPrompt },
      ],
    },
  ],
});

const model = genAI.getGenerativeModel({ model: MODEL });
const result = await model.generateContent([
  { inlineData: { data: imageBase64, mimeType } },
  textPrompt,
]);

const text = result.response.text();
const usage = result.response.usageMetadata;

// promptTokenCount covers both image and text input tokens
generation.end({
  output: text,
  usage: {
    input: usage?.promptTokenCount,
    output: usage?.candidatesTokenCount,
    total: usage?.totalTokenCount,
  },
});

trace.update({ output: text });

console.log("Gemini response:", text);
console.log("Token usage:", usage);

await langfuse.flushAsync();

### View the Trace in Langfuse

Open your [Langfuse Trace Table](https://cloud.langfuse.com). The trace will show:
- The image rendered inline in the generation input
- Combined image + text token counts under **Usage**
- The estimated cost based on the `gemini-2.0-flash` pricing in the Langfuse model registry

## Step 5: Multimodal Tracing in a Next.js Route Handler

In a Next.js application, pass a base64-encoded image from the client. The `flushAsync()` call ensures all Langfuse events are delivered before the serverless function exits.

```ts
// app/api/analyze/route.ts (Next.js App Router)
import { NextRequest, NextResponse } from "next/server";
import { GoogleGenerativeAI } from "@google/generative-ai";
import { Langfuse } from "langfuse";

const langfuse = new Langfuse({
  publicKey: process.env.LANGFUSE_PUBLIC_KEY!,
  secretKey: process.env.LANGFUSE_SECRET_KEY!,
  baseUrl: process.env.LANGFUSE_BASE_URL,
});

const genAI = new GoogleGenerativeAI(process.env.GEMINI_API_KEY!);

export async function POST(req: NextRequest) {
  const { imageBase64, mimeType, prompt } = await req.json();

  const trace = langfuse.trace({
    name: "analyze-image",
    input: { prompt, mimeType },
  });

  const generation = trace.generation({
    name: "gemini-vision",
    model: "gemini-2.0-flash",
    input: [
      {
        role: "user",
        content: [
          {
            type: "image_url",
            image_url: { url: `data:${mimeType};base64,${imageBase64}` },
          },
          { type: "text", text: prompt },
        ],
      },
    ],
  });

  const model = genAI.getGenerativeModel({ model: "gemini-2.0-flash" });
  const result = await model.generateContent([
    { inlineData: { data: imageBase64, mimeType } },
    prompt,
  ]);

  const text = result.response.text();
  const usage = result.response.usageMetadata;

  generation.end({
    output: text,
    usage: {
      input: usage?.promptTokenCount,
      output: usage?.candidatesTokenCount,
      total: usage?.totalTokenCount,
    },
  });
  trace.update({ output: text });

  // Required in serverless — flushes events before the function exits
  await langfuse.flushAsync();

  return NextResponse.json({ text });
}
```

<!-- STEPS_END -->

<!-- MARKDOWN_COMPONENT name: "LearnMore" path: "@/components-mdx/integration-learn-more-js.mdx" -->